In [ ]:

import subprocess, os, shutil, sys
def run(cmd):
    print(">>>", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DATASET = "/kaggle/input/datasets/justrizer/music4all-crs-64k"
WORK = "/kaggle/working/crs_data"
print("dataset exists:", os.path.exists(DATASET), "| files:", len(os.listdir(DATASET)) if os.path.exists(DATASET) else 0, flush=True)

run("pip install -q 'sentence-transformers==6.1.0' 'transformers==5.19.0' lancedb")
run("git clone --depth 1 https://github.com/Hallteon/Yandex-SOTA-HNSW.git repo")
run("cp repo/gemma_crs_pipeline.py .")
run("python -c \"import torch; print('cuda', torch.cuda.device_count())\"")

run(f"python gemma_crs_pipeline.py --work-dir {WORK} --dataset {DATASET} build-texts")
run(f"python gemma_crs_pipeline.py --work-dir {WORK} --dataset {DATASET} embed --embedder gemma --device cuda --dtype float16 --batch-size 8 --max-seq-length 1024 --shard-size 10000")
run(f"python gemma_crs_pipeline.py --work-dir {WORK} --dataset {DATASET} index")
run(f"python gemma_crs_pipeline.py --work-dir {WORK} --dataset {DATASET} evaluate --embedder gemma --device cuda --dtype float16 --max-queries 5000 --batch-size 16")
run(f"python gemma_crs_pipeline.py --work-dir {WORK} --dataset {DATASET} evaluate --embedder gemma --device cuda --dtype float16 --with-profile --max-queries 5000 --batch-size 16")

os.makedirs("/kaggle/working/artifacts", exist_ok=True)
for name in ["lancedb", "vectors", "tracks_texts.jsonl", "manifest.json"]:
    src = f"{WORK}/{name}"
    if os.path.exists(src):
        if os.path.isdir(src):
            shutil.make_archive(f"/kaggle/working/artifacts/{name}", "zip", src)
        else:
            shutil.copy(src, "/kaggle/working/artifacts/")
for f in os.listdir(WORK):
    if f.startswith("eval_"):
        shutil.copy(f"{WORK}/{f}", "/kaggle/working/artifacts/")
print("ARTIFACTS:", sorted(os.listdir("/kaggle/working/artifacts")), flush=True)
